# Pruebas de imputación espacial de contaminantes

## Objetivo

El dataset de calidad del aire presenta **ausencias estructurales** de determinados contaminantes, ya que no todas las estaciones de Madrid miden las cuatro variables seleccionadas para el análisis:

- NO₂
- PM10
- PM2.5
- O₃

Estas ausencias no corresponden necesariamente a fallos puntuales de registro, sino a estaciones que directamente no disponen de sensor para determinados contaminantes.

Para disponer de una representación homogénea de los contaminantes por estación y fecha, se propone evaluar una estrategia de **imputación espacial mediante IDW (Inverse Distance Weighting)**.

In [3]:
# ============================================================
# CARGAR DATASET DE CALIDAD DEL AIRE
# ============================================================

from pathlib import Path
import polars as pl

PROJECT_ROOT = Path.cwd().parents[2]

AIRE_FILE = (
    PROJECT_ROOT
    / "data"
    / "enriched"
    / "fact"
    / "calidad_aire_final.parquet"
)

aire = pl.read_parquet(AIRE_FILE)

print("Dataset cargado correctamente")
print("Shape:", aire.shape)
print("Columnas:", aire.columns)

Dataset cargado correctamente
Shape: (100934, 10)
Columnas: ['provincia', 'municipio', 'estacion', 'magnitud', 'punto_muestreo', 'fecha', 'uom', 'uom_value', 'calidad', 'es_laborable_madrid_ciudad']


## Coordenadas de las estaciones

Para realizar la imputación espacial se utiliza la metadata de las estaciones de calidad del aire, que contiene sus coordenadas geográficas.

Se emplean:

- `ESTACION`: identificador de la estación.
- `LATITUD_G`: latitud en grados decimales.
- `LONGITUD_G`: longitud en grados decimales.

Las estaciones 4 y 11 se excluyen previamente del análisis, de acuerdo con las decisiones tomadas durante el EDA.

In [1]:
from pathlib import Path
import polars as pl
import numpy as np
from math import radians, sin, cos, sqrt, atan2


# ============================================================
# 1. RUTAS
# ============================================================

PROJECT_ROOT = Path.cwd().parents[2]

METADATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "static_files"
    / "metadata_estaciones_aire.csv"
)


# ============================================================
# 2. CARGAR METADATA DE ESTACIONES
# ============================================================

metadata = pl.read_csv(METADATA_FILE)

estaciones_coords = (
    metadata
    .select([
        pl.col("ESTACION").alias("estacion"),
        pl.col("NOMBRE").alias("nombre"),
        pl.col("LATITUD_G").alias("latitud"),
        pl.col("LONGITUD_G").alias("longitud"),
    ])
    .filter(
        ~pl.col("estacion").is_in([4, 11])  # estaciones que hemos decidido eliminar
    )
    .sort("estacion")
)

print("Estaciones con coordenadas:", estaciones_coords.height)

estaciones_coords

Estaciones con coordenadas: 25


estacion,nombre,latitud,longitud
i64,str,f64,f64
8,"""ESCUELAS AGUIRRE""",40.42167,-3.68222
16,"""ARTURO SORIA""",40.44,-3.63917
17,"""VILLAVERDE""",40.34694,-3.705
18,"""FAROLILLO""",40.39472,-3.73194
24,"""CASA DE CAMPO""",40.42,-3.74917
…,…,…,…
57,"""SANCHINARRO""",40.49417,-3.66028
58,"""EL PARDO""",40.51806,-3.77444
59,"""JUAN CARLOS I""",40.465,-3.60889


## Preparación de los datos

Se restringen los datos a **Madrid ciudad** (`provincia = 28`, `municipio = 79`) y a los cuatro contaminantes seleccionados.

Posteriormente, los datos se pivotan para obtener una estructura con una fila por estación y fecha:

| fecha | estación | NO2 | PM10 | PM2.5 | O3 |
|---|---|---|---|---|---|

Los valores nulos permiten identificar los contaminantes que no están disponibles para una determinada estación y fecha.

In [4]:
# ============================================================
# 3. MADRID CIUDAD + 4 CONTAMINANTES
# ============================================================

aire_madrid = (
    aire
    .filter(
        (pl.col("provincia") == 28) &
        (pl.col("municipio") == 79) &
        (~pl.col("estacion").is_in([4, 11])) &
        (pl.col("magnitud").is_in([8, 9, 10, 14]))
    )
    .with_columns(
        pl.when(pl.col("magnitud") == 8).then(pl.lit("NO2"))
        .when(pl.col("magnitud") == 9).then(pl.lit("PM2.5"))
        .when(pl.col("magnitud") == 10).then(pl.lit("PM10"))
        .when(pl.col("magnitud") == 14).then(pl.lit("O3"))
        .alias("contaminante")
    )
)


# ============================================================
# 4. PIVOT
# Una fila = fecha + estación
# ============================================================

aire_pivot = (
    aire_madrid
    .select([
        "fecha",
        "estacion",
        "contaminante",
        "uom_value"
    ])
    .pivot(
        index=["fecha", "estacion"],
        on="contaminante",
        values="uom_value",
        aggregate_function="first"
    )
    .sort(["estacion", "fecha"])
)

aire_pivot

fecha,estacion,NO2,PM2.5,PM10,O3
date,i64,f64,f64,f64,f64
2020-01-01,8,65.666667,28.25,36.208333,5.26625
2020-01-02,8,68.083333,19.083333,29.0,4.294583
2020-01-03,8,73.458333,37.5,50.666667,1.8875
2020-01-04,8,42.347826,6.652174,9.913043,25.96
2020-01-05,8,51.291667,8.041667,14.125,19.327083
…,…,…,…,…,…
2024-12-27,60,25.208333,null,8.25,22.541667
2024-12-28,60,35.916667,null,11.75,19.958333
2024-12-29,60,28.708333,null,10.083333,27.375


## Cálculo de distancias entre estaciones

La distancia entre estaciones se calcula a partir de sus coordenadas geográficas mediante la **fórmula de Haversine**, que permite estimar la distancia sobre la superficie terrestre entre dos puntos definidos por su latitud y longitud.

Para dos estaciones con coordenadas $(\phi_1,\lambda_1)$ y $(\phi_2,\lambda_2)$:

$$
\Delta \phi = \phi_2 - \phi_1
$$

$$
\Delta \lambda = \lambda_2 - \lambda_1
$$

$$
a =
\sin^2\left(\frac{\Delta \phi}{2}\right)
+
\cos(\phi_1)\cos(\phi_2)
\sin^2\left(\frac{\Delta \lambda}{2}\right)
$$

$$
c =
2\arctan2\left(\sqrt{a},\sqrt{1-a}\right)
$$

Finalmente, la distancia entre ambas estaciones es:

$$
d = R \cdot c
$$

donde:

- $\phi$ representa la **latitud** en radianes.
- $\lambda$ representa la **longitud** en radianes.
- $R$ es el radio medio de la Tierra, aproximadamente **6371 km**.
- $d$ es la distancia entre las dos estaciones en kilómetros.

Estas distancias se utilizan posteriormente para determinar qué estaciones son las más cercanas a la estación que se quiere imputar.

In [5]:
# ============================================================
# 5. DISTANCIA HAVERSINE
# ============================================================

def haversine_km(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c

## Imputación mediante IDW

Para estimar los contaminantes que una determinada estación no mide, se utiliza **Inverse Distance Weighting (IDW)**.

La idea del método es que las estaciones más próximas geográficamente a la estación objetivo tengan una mayor influencia sobre el valor estimado que las estaciones más alejadas.

Para cada estación objetivo y fecha se seleccionan las `k` estaciones más cercanas que dispongan de un valor real del contaminante que se desea imputar.

A cada estación vecina se le asigna un peso inversamente proporcional a su distancia:

$$
w_i = \frac{1}{d_i^p}
$$

donde:

- $d_i$ es la distancia entre la estación objetivo y la estación vecina $i$.
- $p$ controla cuánto disminuye la influencia de una estación conforme aumenta su distancia.
- Un valor mayor de $p$ hace que las estaciones más cercanas tengan proporcionalmente mucho más peso.

El valor imputado se obtiene mediante una media ponderada de los valores observados en las `k` estaciones vecinas:

$$
\hat{x} =
\frac{
\sum_{i=1}^{k} w_i x_i
}{
\sum_{i=1}^{k} w_i
}
$$

donde:

- $\hat{x}$ es el valor estimado del contaminante en la estación objetivo.
- $x_i$ es el valor real observado en la estación vecina $i$.
- $w_i$ es el peso asignado a dicha estación.
- $k$ es el número de estaciones vecinas utilizadas.

Por tanto, los dos parámetros que se deben determinar son:

- **`k`**: número de estaciones vecinas utilizadas.
- **`p`**: intensidad con la que la distancia penaliza el peso de las estaciones más alejadas.

Estos parámetros se validan posteriormente utilizando datos reales para seleccionar la configuración que minimiza el error de imputación.

In [6]:
# ============================================================
# 6. FUNCIÓN IDW
# ============================================================

def estimar_idw(
    estacion_objetivo,
    fecha,
    contaminante,
    aire_pivot,
    estaciones_coords,
    k=3,
    p=2
):

    # Coordenadas de la estación objetivo
    target = (
        estaciones_coords
        .filter(
            pl.col("estacion") == estacion_objetivo
        )
    )

    if target.height == 0:
        return None

    lat_obj = target["latitud"][0]
    lon_obj = target["longitud"][0]

    # Estaciones que sí tienen ese contaminante ese día
    disponibles = (
        aire_pivot
        .filter(
            (pl.col("fecha") == fecha) &
            (pl.col("estacion") != estacion_objetivo) &
            (pl.col(contaminante).is_not_null())
        )
        .select([
            "estacion",
            contaminante
        ])
        .join(
            estaciones_coords,
            on="estacion",
            how="left"
        )
        .drop_nulls([
            "latitud",
            "longitud"
        ])
    )

    if disponibles.height == 0:
        return None

    vecinos = []

    for fila in disponibles.iter_rows(named=True):

        distancia = haversine_km(
            lat_obj,
            lon_obj,
            fila["latitud"],
            fila["longitud"]
        )

        vecinos.append({
            "estacion": fila["estacion"],
            "valor": fila[contaminante],
            "distancia": distancia
        })

    # Ordenar por cercanía
    vecinos = sorted(
        vecinos,
        key=lambda x: x["distancia"]
    )[:k]

    if len(vecinos) == 0:
        return None

    # Evitar división por cero
    if vecinos[0]["distancia"] == 0:
        return vecinos[0]["valor"]

    pesos = np.array([
        1 / (v["distancia"] ** p)
        for v in vecinos
    ])

    valores = np.array([
        v["valor"]
        for v in vecinos
    ])

    estimacion = (
        np.sum(pesos * valores)
        / np.sum(pesos)
    )

    return float(estimacion)

## Validación y selección de parámetros

Los parámetros `k` y `p` no se seleccionan de forma arbitraria. Se realiza una validación utilizando observaciones reales del dataset.

Se aplica una estrategia **leave-one-station-out**:

1. Se selecciona una estación que dispone de observaciones reales de un contaminante.
2. Para cada fecha, se estima su valor utilizando únicamente las demás estaciones que disponen de dicho contaminante.
3. La estimación obtenida mediante IDW se compara con el valor real observado.
4. El procedimiento se repite para todas las estaciones y observaciones disponibles.
5. Se prueban diferentes combinaciones de `k` y `p`.

La validación se realiza de forma independiente para **PM2.5, PM10 y O₃**, ya que son los contaminantes que presentan ausencias estructurales entre estaciones.

Se prueban:

- `k = 2, 3, 4, 5, 6, 7`
- `p = 1, 1.5, 2, 2.5, 3`

Para cada combinación se calculan:

- **MAE**: error absoluto medio.
- **RMSE**: raíz del error cuadrático medio.
- **R²**: capacidad de la imputación para reproducir la variabilidad de los valores observados.

Como criterio principal de selección se utiliza el **menor RMSE**, aunque se revisan también MAE y R².

El objetivo es obtener una configuración de IDW validada empíricamente para cada contaminante antes de incorporar la imputación al pipeline definitivo.

In [12]:
# ============================================================
# VALIDACIÓN GENERAL IDW
# Leave-One-Station-Out
# PM2.5 + PM10 + O3
# ============================================================

import numpy as np
import polars as pl

contaminantes_validar = ["PM2.5", "PM10", "O3"]

valores_k = [2, 3, 4, 5, 6, 7]
valores_p = [1, 1.5, 2, 2.5, 3]

resultados_generales = []


for contaminante in contaminantes_validar:

    print("\n" + "=" * 80)
    print(f"VALIDANDO {contaminante}")
    print("=" * 80)

    # --------------------------------------------------------
    # Estaciones que realmente miden ese contaminante
    # --------------------------------------------------------

    estaciones_validas = (
        aire_pivot
        .filter(pl.col(contaminante).is_not_null())
        .select("estacion")
        .unique()
        .sort("estacion")
        ["estacion"]
        .to_list()
    )

    print("Estaciones:", estaciones_validas)
    print("Nº estaciones:", len(estaciones_validas))


    # --------------------------------------------------------
    # GRID SEARCH
    # --------------------------------------------------------

    for k in valores_k:

        for p in valores_p:

            reales_totales = []
            estimados_totales = []

            # Leave-one-station-out
            for estacion in estaciones_validas:

                datos_estacion = (
                    aire_pivot
                    .filter(
                        (pl.col("estacion") == estacion) &
                        (pl.col(contaminante).is_not_null())
                    )
                    .select([
                        "fecha",
                        contaminante
                    ])
                )

                for fila in datos_estacion.iter_rows(named=True):

                    estimado = estimar_idw(
                        estacion_objetivo=estacion,
                        fecha=fila["fecha"],
                        contaminante=contaminante,
                        aire_pivot=aire_pivot,
                        estaciones_coords=estaciones_coords,
                        k=k,
                        p=p
                    )

                    if estimado is not None:

                        reales_totales.append(
                            fila[contaminante]
                        )

                        estimados_totales.append(
                            estimado
                        )

            # ------------------------------------------------
            # MÉTRICAS
            # ------------------------------------------------

            reales = np.array(reales_totales)
            estimados = np.array(estimados_totales)

            mae = np.mean(
                np.abs(reales - estimados)
            )

            rmse = np.sqrt(
                np.mean((reales - estimados) ** 2)
            )

            r2 = (
                1
                - np.sum((reales - estimados) ** 2)
                / np.sum(
                    (reales - np.mean(reales)) ** 2
                )
            )

            resultados_generales.append({
                "contaminante": contaminante,
                "k": k,
                "p": p,
                "n_observaciones": len(reales),
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2
            })

            print(
                f"{contaminante} | "
                f"k={k} p={p} | "
                f"MAE={mae:.3f} | "
                f"RMSE={rmse:.3f} | "
                f"R2={r2:.3f}"
            )


# ============================================================
# RESULTADOS
# ============================================================

resultados_idw = pl.DataFrame(
    resultados_generales
)


# ============================================================
# MEJOR CONFIGURACIÓN POR CONTAMINANTE
# ============================================================

mejores = (
    resultados_idw
    .sort([
        "contaminante",
        "RMSE"
    ])
    .group_by(
        "contaminante",
        maintain_order=True
    )
    .first()
)


print("\n")
print("=" * 80)
print("MEJOR CONFIGURACIÓN POR CONTAMINANTE")
print("=" * 80)

with pl.Config(
    tbl_rows=-1,
    tbl_cols=-1,
    fmt_str_lengths=100
):
    print(mejores)


# ============================================================
# TODAS LAS CONFIGURACIONES
# ============================================================

print("\n")
print("=" * 80)
print("RESULTADOS COMPLETOS")
print("=" * 80)

with pl.Config(
    tbl_rows=-1,
    tbl_cols=-1,
    fmt_str_lengths=100
):
    print(
        resultados_idw.sort([
            "contaminante",
            "RMSE"
        ])
    )


VALIDANDO PM2.5
Estaciones: [8, 24, 38, 47, 48, 50, 56, 57]
Nº estaciones: 8
PM2.5 | k=2 p=1 | MAE=2.443 | RMSE=3.884 | R2=0.602
PM2.5 | k=2 p=1.5 | MAE=2.446 | RMSE=3.896 | R2=0.600
PM2.5 | k=2 p=2 | MAE=2.452 | RMSE=3.913 | R2=0.596
PM2.5 | k=2 p=2.5 | MAE=2.460 | RMSE=3.934 | R2=0.592
PM2.5 | k=2 p=3 | MAE=2.470 | RMSE=3.955 | R2=0.587
PM2.5 | k=3 p=1 | MAE=2.296 | RMSE=3.624 | R2=0.654
PM2.5 | k=3 p=1.5 | MAE=2.310 | RMSE=3.655 | R2=0.648
PM2.5 | k=3 p=2 | MAE=2.328 | RMSE=3.693 | R2=0.640
PM2.5 | k=3 p=2.5 | MAE=2.349 | RMSE=3.737 | R2=0.632
PM2.5 | k=3 p=3 | MAE=2.373 | RMSE=3.782 | R2=0.623
PM2.5 | k=4 p=1 | MAE=2.253 | RMSE=3.540 | R2=0.669
PM2.5 | k=4 p=1.5 | MAE=2.269 | RMSE=3.579 | R2=0.662
PM2.5 | k=4 p=2 | MAE=2.292 | RMSE=3.626 | R2=0.653
PM2.5 | k=4 p=2.5 | MAE=2.318 | RMSE=3.680 | R2=0.643
PM2.5 | k=4 p=3 | MAE=2.346 | RMSE=3.734 | R2=0.632
PM2.5 | k=5 p=1 | MAE=2.168 | RMSE=3.434 | R2=0.689
PM2.5 | k=5 p=1.5 | MAE=2.195 | RMSE=3.484 | R2=0.680
PM2.5 | k=5 p=2 | MAE=2.

## Resultados de la validación IDW

Tras realizar la validación *leave-one-station-out* para los contaminantes **PM2.5, PM10 y O₃**, se selecciona para cada contaminante la combinación de parámetros `(k, p)` que presenta el **menor RMSE**.

La validación se ha realizado utilizando todas las observaciones reales disponibles de las estaciones que miden cada contaminante. Para cada estación se estima su valor a partir de las restantes estaciones y posteriormente se compara la estimación con el valor real observado.

Los mejores resultados obtenidos son:

| Contaminante | k | p | Nº observaciones | MAE | RMSE | R² |
|---|---:|---:|---:|---:|---:|---:|
| **PM2.5** | **7** | **1.0** | 14.040 | 2.109 | **3.355** | 0.703 |
| **PM10** | **7** | **1.5** | 23.421 | 3.990 | **6.415** | 0.818 |
| **O₃** | **7** | **1.0** | 23.524 | 4.797 | **6.288** | 0.930 |

### Interpretación

Los resultados muestran que el uso de **7 estaciones vecinas (`k = 7`)** proporciona el menor RMSE para los tres contaminantes analizados.

El parámetro de ponderación de la distancia (`p`) presenta una pequeña diferencia entre contaminantes:

- **PM2.5:** `k = 7`, `p = 1.0`
- **PM10:** `k = 7`, `p = 1.5`
- **O₃:** `k = 7`, `p = 1.0`

Por tanto, para PM2.5 y O₃ el peso de cada estación será inversamente proporcional a la distancia:

$$
w_i = \frac{1}{d_i}
$$

mientras que para PM10 se utilizará:

$$
w_i = \frac{1}{d_i^{1.5}}
$$

En todos los casos, el valor imputado se calculará utilizando las **7 estaciones más cercanas que dispongan de un valor real del contaminante en la fecha correspondiente**.

### Calidad de la imputación

Los valores de R² obtenidos muestran diferente capacidad de reconstrucción espacial dependiendo del contaminante:

- **O₃: R² = 0.930**, mostrando una elevada capacidad para reproducir los valores observados mediante información de estaciones cercanas.
- **PM10: R² = 0.818**, mostrando también un buen comportamiento de la interpolación espacial.
- **PM2.5: R² = 0.703**, presentando una capacidad de reconstrucción inferior a los anteriores, aunque manteniendo una relación considerable entre los valores estimados y observados.

Estos resultados permiten seleccionar los parámetros de IDW de forma empírica, en lugar de establecerlos de forma arbitraria.

## Parámetros propuestos para la imputación final

La implementación de la imputación en el pipeline utilizará:

| Contaminante | k | p |
|---|---:|---:|
| PM2.5 | 7 | 1.0 |
| PM10 | 7 | 1.5 |
| O₃ | 7 | 1.0 |

Para cada ausencia estructural se identificarán las estaciones que dispongan de una observación real del contaminante en esa fecha, se seleccionarán las **7 más cercanas** y se aplicará IDW utilizando el parámetro `p` correspondiente.

Se recomienda conservar adicionalmente un indicador que permita distinguir los valores originales de los valores obtenidos mediante imputación, por ejemplo:

`es_imputado = True / False`

## Implementación propuesta en el pipeline

Una vez seleccionados los parámetros óptimos de IDW para cada contaminante, la imputación deberá incorporarse al pipeline de calidad del aire.

### 1. Cobertura de contaminantes

El archivo `cobertura_contaminantes_estaciones_madrid.xlsx` identifica qué contaminantes son medidos originalmente por cada estación y en qué años.

Las estaciones **4 y 11 quedan excluidas del dataset de modelado**.

Las ausencias estructurales de PM10, PM2.5 y O3 deberán completarse mediante imputación espacial.

### 2. Lógica de imputación

Para cada combinación:

`fecha + estación + contaminante`

en la que el contaminante no sea medido por la estación:

1. Obtener las coordenadas de la estación objetivo desde `metadata_estaciones_aire.csv`.
2. Identificar las estaciones que sí disponen de una observación real de ese contaminante en esa fecha.
3. Calcular la distancia entre la estación objetivo y las estaciones disponibles mediante Haversine.
4. Ordenarlas por distancia.
5. Seleccionar las `k` estaciones más cercanas.
6. Calcular el valor mediante IDW utilizando el parámetro `p` seleccionado para ese contaminante.
7. Incorporar el valor estimado al dataset final.

Los parámetros `k` y `p` utilizados serán los obtenidos mediante la validación realizada en este notebook.

### 3. Trazabilidad

Se recomienda conservar una variable que permita distinguir entre valores originales e imputados, por ejemplo:

`es_imputado = True / False`

De esta forma será posible identificar posteriormente qué observaciones proceden directamente de las estaciones y cuáles han sido estimadas mediante IDW.